In [1]:
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
import pickle

In [2]:
from tensorflow.compat.v1 import ConfigProto
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.8
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
with open('data.pkl', 'rb') as f:
    [x_train,y_train,x_test,y_test]=pickle.load(f)
y_train_onehot=tf.keras.utils.to_categorical(y_train,num_classes=4)
y_test_onehot=tf.keras.utils.to_categorical(y_test,num_classes=4)

In [4]:
from load_network import load_VGG, load_Res, load_ViT, load_Mix
from noise import PGDClassificationAdversaryL2
from train import WarmUpCosine, CustomWeightDecaySGD, AdamW
from network import HierarchicalConvEmbedding, TransformerBlock, AddPositionEmbedding, MLP, MixerBlock

In [5]:
# ============================================================
# 32x32 resolution: recommended starting parameters (empirical values)
# ============================================================
def suggested_l2_eps_for_32x32():
    """
    For [0,1] normalized 32x32 images:
      - Light perturbation: eps ≈ 0.5 ~ 1.5
      - Medium strength: eps ≈ 2.0 ~ 3.0
      - Stronger perturbation: eps ≈ 4.0+ (may be noticeable)
    You can start with eps=2.0 for attack strength curves.
    """
    return {"light": (0.5, 1.5), "medium": (2.0, 3.0), "strong": (4.0, 6.0)}

In [7]:
def attack_models(model):
    #attacker = PatchPGDClassificationAdversaryL2(model, num_classes=4,from_logits=False, clip_min=0.0, clip_max=1.0)
    attacker = PGDClassificationAdversaryL2(model, num_classes=4, from_logits=False, clip_min=0.0, clip_max=1.0)
    x_adv = attacker.generate_adversarial_dataset(x_train, tf.squeeze(y_train, axis=-1), eps=2.0, alpha=0.2, steps=20,
                                  batch_size=128,
                                  shuffle=True,
                                  random_start=True)
                                  #patch_size=8,   # 与 ViT patch size 对齐
                                  #patch_k=16)

    return x_adv

In [12]:
NOISE_DIR = "./attack/"

In [13]:
def save_wb_per_layer(NOISE_DIR, N=5):
    for i in range(N):
        NOISE_I_DIR = NOISE_DIR + str(i)
        os.makedirs(NOISE_I_DIR, exist_ok=True)
        x_pgd_VGG= attack_models(model_VGG)
        model_VGG.evaluate(x_pgd_VGG,y_train_onehot)
        np.save(os.path.join(NOISE_I_DIR, "VGG_pgd.npy"), x_pgd_VGG, allow_pickle=True)
        x_pgd_Res = attack_models(model_Res)
        model_Res.evaluate(x_pgd_Res,y_train_onehot)
        np.save(os.path.join(NOISE_I_DIR, "Res_pgd.npy"), x_pgd_Res, allow_pickle=True)
        x_pgd_ViT = attack_models(model_ViT)
        model_ViT.evaluate(x_pgd_ViT,y_train_onehot)
        np.save(os.path.join(NOISE_I_DIR, "ViT_pgd.npy"), x_pgd_ViT, allow_pickle=True)
        x_pgd_Mix = attack_models(model_Mix)
        model_Mix.evaluate(x_pgd_Mix,y_train_onehot)
        np.save(os.path.join(NOISE_I_DIR, "Mix_pgd.npy"), x_pgd_Mix, allow_pickle=True)
    print("Saved all noise data into separate files.")

In [14]:
model_VGG=load_VGG()
model_Res=load_Res()
model_ViT=load_ViT()
model_Mix=load_Mix()
save_wb_per_layer(NOISE_DIR)

625/625 [==============================] - 11s 17ms/step - loss: 9.0235 - accuracy: 1.5000e-04
Saved all noise data into separate files.
